# 12 — Demand Uncertainty

Estimate predictive uncertainty from **out-of-sample validation residuals**, not training predictions.

In [ ]:

from pathlib import Path
import os, sys
ROOT = Path.cwd()
while not (ROOT / "README.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("Project root:", ROOT)


In [ ]:
import joblib, pandas as pd
stats=joblib.load(ROOT/"models/uncertainty/residual_stats.joblib"); display(pd.Series(stats).to_frame("value"))

In [ ]:
import matplotlib.pyplot as plt, numpy as np
df=pd.read_parquet(ROOT/"data/processed/forecast_features.parquet"); cfg=__import__("src.utils.config",fromlist=["load_yaml"]).load_yaml("model_config.yaml"); maxd=df.date.max(); test_start=maxd-pd.Timedelta(days=cfg["test_days"]-1); val_start=test_start-pd.Timedelta(days=cfg["validation_days"]); val=df[(df.date>=val_start)&(df.date<test_start)]; bundle=joblib.load(ROOT/"models/forecasting/validation_model.joblib"); pred=bundle["model"].predict(val[bundle["feature_columns"]]).clip(min=0); residual=val.demand.to_numpy()-pred; fig,ax=plt.subplots(figsize=(8,4)); ax.hist(residual,bins=40); ax.set_title("Out-of-sample validation residuals"); plt.show()

Residual mean close to zero suggests little systematic bias; residual spread determines the width of the forecast interval. Inventory safety stock will use the documented daily-demand approximation once, through the protection-period square-root scaling.